<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_4_ReinforcementLearning/student/Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_4_ReinforcementLearning/student/Tutorial2.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Tutorial 2: Temporal-difference learning and dopamine

**Session 4: Reinforcement Learning**

This tutorial reproduces the classic result of Schultz, Dayan & Montague (1997): the phasic firing of midbrain dopamine neurons behaves like the temporal-difference (TD) **reward prediction error** (RPE)

$$\delta_t = r_t + \gamma\, V(t{+}1) - V(t).$$

Running a plain **TD(0)** learner on a simple cue→reward conditioning task recovers the three hallmark dopamine responses:

| # | Condition | RPE signature |
|---|-----------|---------------|
| 1 | Naive cue→reward pairing | spikes at **reward** time |
| 2 | After learning | spikes at **cue onset**, ~0 at reward |
| 3 | Reward omitted | dips **negative** at the expected reward time |

The agent uses the *complete serial compound* (CSC) representation: every timestep after the cue is its own state, so value is learned as a function of time since the cue, and the prediction error can migrate backwards in time.

A bonus section at the end contrasts model-free and model-based values after reward devaluation (Part 3 of the lecture).

*Adapted from the nabi-RL hands-on demo.*

## Setup

In [ ]:
# @title Imports
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 1. Task definition

Each trial consists of `N_STEPS` discrete timesteps, modelling a classical conditioning paradigm:

- **`CUE_T`**: the timestep at which the conditioned stimulus (CS) appears
- **`REWARD_T`**: the timestep at which the reward is delivered

The agent's goal is to learn to **predict the expected future reward** from each state.

In [ ]:
N_STEPS = 16        # total number of timesteps in one trial
CUE_T = 4           # timestep at which the conditioned stimulus appears
REWARD_T = 10       # timestep at which reward is delivered
REWARD_SIZE = 1.0   # magnitude of reward


def make_trial(rewarded=True):
    """Return the reward vector r_t for a single trial. r_t = REWARD_SIZE at t=REWARD_T, 0 everywhere else."""
    r = np.zeros(N_STEPS)
    if rewarded:
        r[REWARD_T] = REWARD_SIZE
    return r

## 2. TD(0) learner (complete serial compound)

State `s` = timestep index. The agent learns a value function $V(t)$, the expected future reward from each timestep.

**Complete serial compound (CSC) representation.** Every timestep after the cue is treated as its own distinct state. This allows the prediction error to migrate backwards in time, from the reward timestep toward the cue.

Before the cue ($t <$ `CUE_T`), the agent is in a null state with $V = 0$: no prediction is made until the cue appears, because the cue arrives at an unpredictable time.

At each timestep the TD error is

$$\delta_t = r_t + \gamma V(t+1) - V(t),$$

and the value of the current state is updated by a small step in the direction of $\delta_t$: $V(t) \leftarrow V(t) + \alpha\,\delta_t$.

In [ ]:
class TDLearner:
    def __init__(self, alpha=0.3, gamma=0.98):
        self.alpha = alpha          # learning rate: how fast V is updated
        self.gamma = gamma          # discount factor: how much future rewards are discounted
        self.V = np.zeros(N_STEPS)  # value function, initialized to 0

    def value(self, t):
        if t < CUE_T or t >= N_STEPS:
            return 0.0  # before the cue or after the trial ends: null state, V = 0
        return float(self.V[t])

    def run_trial(self, r, learn=True):
        """Play one trial, return the per-timestep RPE delta_t."""
        deltas = np.zeros(N_STEPS)
        for t in range(N_STEPS):
            v_t = self.value(t)
            v_next = self.value(t + 1)
            delta = r[t] + self.gamma * v_next - v_t  # TD error
            deltas[t] = delta
            if learn and CUE_T <= t < N_STEPS:        # update V only after the cue (CSC representation)
                self.V[t] += self.alpha * delta
        return deltas

## 3. Run the experiment

We run the experiment in three phases:

1. **Naive trial**: record the RPE trace before any learning has occurred.
2. **Training**: run `n_train` rewarded trials with learning enabled, until $V$ converges.
3. **Probe trials** (no learning): observe the learned RPE pattern.
    - **Rewarded probe**: reward is delivered as usual, so the RPE should now peak at the cue.
    - **Omission probe**: reward is omitted, so the RPE should dip negative at the expected reward time.

In [ ]:
def run_experiment(n_train=200, seed=0):
    agent = TDLearner()

    # Phase 1 - Naive trial: record RPE before any learning
    naive_delta = agent.run_trial(make_trial(rewarded=True), learn=True).copy()

    # Phase 2 - Training: run n_train rewarded trials with learning enabled
    delta_history = [naive_delta]
    for _ in range(n_train - 1):
        d = agent.run_trial(make_trial(rewarded=True), learn=True)
        delta_history.append(d.copy())
    trained_delta = delta_history[-1]   # RPE trace of the final training trial

    # Phase 3 - Probe trials: observe learned RPE pattern (no learning)
    # Probe 1: reward delivered -> RPE should now peak at cue onset
    probe_reward = agent.run_trial(make_trial(rewarded=True), learn=False)
    # Probe 2: reward omitted -> RPE should dip negative at expected reward time
    probe_omit = agent.run_trial(make_trial(rewarded=False), learn=False)

    return {
        "agent": agent,
        "naive_delta": naive_delta,
        "trained_delta": trained_delta,
        "delta_history": np.array(delta_history),
        "probe_reward": probe_reward,
        "probe_omit": probe_omit,
    }


res = run_experiment()
np.round(res["agent"].V, 3)   # inspect the learned value function

```{admonition} Think!
The learned values rise from the cue toward the reward instead of being flat. Why? Relate the ratio between neighbouring values to $\gamma$.
```

## 4. The three RPE signatures

> **Note:** the cue-onset RPE lands at `CUE_T - 1` rather than `CUE_T`. With the convention $\delta_t = r_t + \gamma V(t+1) - V(t)$, the error is computed on the step *into* the cue state.

The cell below prints a numerical summary of the three RPE signatures.

In [ ]:
naive = res["naive_delta"]
probe = res["probe_reward"]
omit = res["probe_omit"]
cue_onset = CUE_T - 1

print(f"cue at t={CUE_T}, reward at t={REWARD_T}\n")
print("1) Naive trial   -> RPE peaks at REWARD time")
print(f"   argmax delta  = t={int(np.argmax(naive))}  (delta={naive.max():+.3f})")
print("\n2) After learning (rewarded probe) -> RPE peaks at CUE onset")
print(f"   delta at cue onset (t={cue_onset})  = {probe[cue_onset]:+.3f}")
print(f"   delta at reward    (t={REWARD_T}) = {probe[REWARD_T]:+.3f}")
print("\n3) Reward omission probe -> RPE dips NEGATIVE at expected reward")
print(f"   delta at reward (t={REWARD_T}) = {omit[REWARD_T]:+.3f}")

In [ ]:
t = np.arange(N_STEPS)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)

panels = [
    ("Naive: RPE at reward", naive, "tab:blue"),
    ("Learned: RPE at cue", probe, "tab:green"),
    ("Omission: negative dip", omit, "tab:red"),
]

for ax, (title, delta, color) in zip(axes, panels):
    ax.axhline(0, color="0.7", lw=0.8)
    ax.bar(t, delta, color=color, width=0.8)  # RPE as bar plot
    ax.axvline(CUE_T, color="0.5", ls=":", lw=1, label="cue")         # cue onset
    ax.axvline(REWARD_T, color="0.5", ls="--", lw=1, label="reward")  # reward time
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("timestep")
    ax.legend(fontsize=8)

axes[0].set_ylabel("RPE  delta_t")
fig.suptitle("Dopamine reward prediction error reproduced by TD(0)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 5. The RPE peak migrating from reward to cue

The heatmap below shows how the RPE evolves over the course of training:

- **x-axis**: timestep within a trial
- **y-axis**: trial number (0 = naive, `n_train` = fully trained)
- **color**: RPE value (red = positive, blue = negative)

Early in training, the RPE spike sits at the reward timestep. As learning progresses, it marches backward in time and settles at the cue onset, reproducing the hallmark dopamine response observed by Schultz et al. (1997).

In [ ]:
hist = res["delta_history"]  # shape: (n_train, N_STEPS) - RPE trace for every trial

vmin = -np.abs(hist).max()  # symmetric color scale around 0
vmax = np.abs(hist).max()

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(hist, aspect="auto", origin="lower", cmap="RdBu_r",
               vmin=vmin, vmax=vmax, extent=(0, N_STEPS, 0, hist.shape[0]))

ax.axvline(CUE_T, color="k", ls=":", lw=1, label="cue")
ax.axvline(REWARD_T, color="k", ls="--", lw=1, label="reward")
ax.set_xlabel("timestep")
ax.set_ylabel("trial")
ax.set_title("RPE peak migrates from reward -> cue")
ax.legend(loc="upper right", fontsize=8)
fig.colorbar(im, ax=ax, label="delta_t")
fig.tight_layout()
plt.show()

## 6. RPE trace trial by trial

The panels below show single trials along the way. Early trials show the RPE peaking at the reward; later trials show it migrating to the cue.

In [ ]:
def plot_rpe_by_trial(trial, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 3))
    ax.axhline(0, color="0.7", lw=0.8)
    ax.bar(np.arange(N_STEPS), hist[trial], color="tab:purple", width=0.8)  # RPE as bar plot
    ax.axvline(CUE_T, color="0.5", ls=":", lw=1, label="cue")
    ax.axvline(REWARD_T, color="0.5", ls="--", lw=1, label="reward")
    ax.set_xlabel("timestep")
    ax.set_title(f"trial {trial}")
    return ax


snapshots = [0, 3, 10, 30, len(hist) - 1]
fig, axes = plt.subplots(1, len(snapshots), figsize=(14, 2.8), sharey=True)
for ax, trial in zip(axes, snapshots):
    plot_rpe_by_trial(trial, ax)
axes[0].set_ylabel("RPE delta_t")
fig.tight_layout()
plt.show()

### Interactive RPE explorer

When you run this notebook yourself (for example in Colab), the slider below lets you step through every training trial.

In [ ]:
from ipywidgets import IntSlider, interact


@interact(trial=IntSlider(min=0, max=len(hist) - 1, step=1, value=0, description="Trial #"))
def explore_rpe(trial):
    ax = plot_rpe_by_trial(trial)
    ax.set_ylabel("RPE delta_t")
    ax.legend(fontsize=8)
    plt.show()

```{admonition} Think!
- Set `alpha=1.0, gamma=1.0` in `TDLearner` and work through the first three trials by hand, as in the lecture's *How the Burst Moves to the Cue* slide. How many trials does it take the burst to reach the cue?
- What happens to the omission dip if the reward is delivered on only half of the training trials?
```

## Bonus: model-free vs. model-based values after devaluation

The TD learner above is **model-free**: it caches values directly from experience and never represents *where* its actions lead. A **model-based** agent instead stores a transition model $T(s'\mid s,a)$ and reward $R$, and computes values by planning:

$$Q(s,a) = R(s,a) + \gamma \sum_{s'} T(s'\mid s,a)\, V(s').$$

The lecture's example: from home ($s$), the action *Go* reaches the café with probability 0.7 and gets caught in the rain with probability 0.3. The café is worth 10, the rain 0.
Then the café is **devalued** (you are already full, so it is worth only 2). The agent learns this from one taste, without walking from home again.

In [ ]:
T_go = {"cafe": 0.7, "rain": 0.3}       # learned transition model T(s' | home, Go)
R_go = 0.0                              # immediate reward for Go
gamma = 1.0
outcome_value = {"cafe": 10.0, "rain": 0.0}


def plan_q(outcome_value):
    """Model-based: compute Q(home, Go) from the model with a one-step Bellman backup."""
    return R_go + gamma * sum(p * outcome_value[s] for s, p in T_go.items())


# Both systems agree before devaluation: the model-free cache was learned from many trips.
q_model_based = plan_q(outcome_value)
q_model_free = q_model_based
print(f"Before devaluation:  model-based Q = {q_model_based:.1f},  model-free Q = {q_model_free:.1f}")

# Devaluation: the cafe is now worth 2. Only the outcome value changes.
outcome_value["cafe"] = 2.0
q_model_based = plan_q(outcome_value)   # re-plan immediately from the model
print(f"After devaluation:   model-based Q = {q_model_based:.1f},  model-free Q = {q_model_free:.1f} (stale)")


# The model-free cache only catches up through new experience of Go.
rng = np.random.default_rng(1)
alpha = 0.2
trace = [q_model_free]
for _ in range(30):
    outcome = "cafe" if rng.random() < T_go["cafe"] else "rain"
    q_model_free += alpha * (R_go + gamma * outcome_value[outcome] - q_model_free)
    trace.append(q_model_free)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(trace, marker="o", ms=3, label="model-free (TD updates)")
ax.axhline(q_model_based, color="tab:green", ls="--", label="model-based (re-planned)")
ax.set_xlabel("new trips from home after devaluation")
ax.set_ylabel("Q(home, Go)")
ax.legend()
plt.show()

Animals show both behaviours. Early in training, responding drops right after devaluation (goal-directed, model-based). After extensive training, it persists (habitual, model-free). The dorsomedial and dorsolateral striatum are thought to support these two modes, respectively.

## References

- Schultz, W., Dayan, P. & Montague, P. R. (1997). A neural substrate of prediction and reward. *Science*, 275(5306), 1593–1599.
- Sutton, R. S. & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.), Chapters 6 and 8. MIT Press.
- Daw, N. D., Niv, Y. & Dayan, P. (2005). Uncertainty-based competition between prefrontal and dorsolateral striatal systems for behavioral control. *Nature Neuroscience*, 8(12), 1704–1711.